In [ ]:
n = 4
constraints = [(0, 1, 5, 6), (0, 2, 5, 7), (0, 3, 5, 7)]

Solución A: Log-ratios

In [2]:
import numpy as np

# Entradas:
# n: número de incógnitas X1..Xn
# constraints: lista de tuplas (i, j, ki, kj) para ecuaciones Xi*ki = Xj*kj
# weights: opcional, misma longitud que constraints

def solve_log_ratios(n, constraints, weights=None, X1_anchor=1.0):
    m = len(constraints)
    B = np.zeros((m, n))
    b = np.zeros(m)
    for r, (i, j, ki, kj) in enumerate(constraints):
        # i, j en [0..n-1]
        B[r, i] =  1.0
        B[r, j] = -1.0
        b[r] = np.log(kj) - np.log(ki)

    if weights is None:
        W = np.eye(m)
    else:
        W = np.diag(np.asarray(weights))

    # Anclamos a1 = log(X1_anchor)
    # Quitamos la columna 0 y ajustamos el término independiente
    B_free = B[:, 1:]
    b_tilde = b - B[:, 0]*np.log(X1_anchor)

    # WLS
    BtWB = B_free.T @ W @ B_free
    BtWb = B_free.T @ W @ b_tilde
    a_free = np.linalg.solve(BtWB, BtWb)

    a = np.zeros(n)
    a[0] = np.log(X1_anchor)
    a[1:] = a_free

    # IC (opcional)
    resid = b - B @ a
    dof = m - (n - 1)
    s2 = (resid.T @ W @ resid) / max(dof, 1)
    Cov_free = s2 * np.linalg.inv(BtWB)
    se = np.zeros(n)
    se[0] = 0.0
    se[1:] = np.sqrt(np.diag(Cov_free))

    X_hat = np.exp(a)
    # IC 95% en escala original
    lo = np.exp(a - 1.96*se)
    hi = np.exp(a + 1.96*se)
    return X_hat, (lo, hi), a, se


In [ ]:
import numpy as np

def solve_homogeneous(n, constraints, X1_anchor=1.0):
    rows = []
    for (i, j, ki, kj) in constraints:
        row = np.zeros(n)
        row[i] =  ki
        row[j] = -kj
        rows.append(row)
    A = np.vstack(rows)

    # SVD
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    x_rel = Vt[-1]             # vector asociado al s.v. más pequeño
    # Reescalar para fijar X1 = X1_anchor
    factor = X1_anchor / x_rel[0]
    X_hat = factor * x_rel
    # Si quieres positividad, multiplica por -1 si la mayor parte sale negativa
    if np.sum(X_hat > 0) < np.sum(X_hat < 0):
        X_hat = -X_hat
    return X_hat


In [3]:
solve_log_ratios(n, constraints)

(array([1.        , 0.83333333, 0.71428571, 0.71428571]),
 (array([1.        , 0.83333333, 0.71428571, 0.71428571]),
  array([1.        , 0.83333333, 0.71428571, 0.71428571])),
 array([ 0.        , -0.18232156, -0.33647224, -0.33647224]),
 array([0., 0., 0., 0.]))

Solución B: SVD

In [4]:
import numpy as np

def solve_homogeneous(n, constraints, X1_anchor=1.0):
    rows = []
    for (i, j, ki, kj) in constraints:
        row = np.zeros(n)
        row[i] =  ki
        row[j] = -kj
        rows.append(row)
    A = np.vstack(rows)

    # SVD
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    x_rel = Vt[-1]             # vector asociado al s.v. más pequeño
    # Reescalar para fijar X1 = X1_anchor
    factor = X1_anchor / x_rel[0]
    X_hat = factor * x_rel
    # Si quieres positividad, multiplica por -1 si la mayor parte sale negativa
    if np.sum(X_hat > 0) < np.sum(X_hat < 0):
        X_hat = -X_hat
    return X_hat


In [5]:
solve_homogeneous(n, constraints)

array([ 1.        , -7.76651672,  3.83046809,  3.83046809])